# AI-Generated Image Detection

This notebook trains models to detect AI-generated images including:
- GAN-generated images (StyleGAN, ProGAN, etc.)
- Diffusion model images (DALL-E, Midjourney, Stable Diffusion)
- Deepfake faces

## Model Architecture
Uses EfficientNet-B4 as backbone with transfer learning for better accuracy.


## Imports and Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from PIL import Image
from pathlib import Path

%matplotlib inline
np.random.seed(42)

# Deep Learning
import tensorflow as tf
from keras.applications import EfficientNetB4
from keras.layers import Dense, Dropout, GlobalAveragePooling2D
from keras.models import Model
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from keras.utils import to_categorical

# Data handling
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import itertools

sns.set(style='white', context='notebook', palette='deep')


SyntaxError: invalid syntax (2171313771.py, line 1)

## Configuration


In [23]:
# Configuration
IMG_SIZE = (224, 224)  # EfficientNet input size
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.0001

# Paths
DATASET_DIR = 'datasets/ai_detection'
MODELS_DIR = 'models/ai_generated'

# Create directories
os.makedirs(MODELS_DIR, exist_ok=True)

# Class mapping
# 0 = Real image
# 1 = AI-generated (GAN)
# 2 = AI-generated (Diffusion)
NUM_CLASSES = 3  # Real, GAN, Diffusion
CLASS_NAMES = ['Real', 'GAN', 'Diffusion']


## Load Saved Model (Optional)

Check if a trained model exists. If found, you can skip training and use it directly for inference.


In [24]:
# I have already trained the model and it took 13 hours to train the model so i have simply saved it and loading it here
# Load saved model if it exists (skip training if model is already trained)
from keras.models import load_model

best_model_path = os.path.join(MODELS_DIR, 'ai_detector_best.keras')
final_model_path = os.path.join(MODELS_DIR, 'ai_detector_final.keras')

model = None
model_loaded = False

# Try to load best model first, then final model
if os.path.exists(best_model_path):
    print("✓ Found saved best model! Loading...")
    model = load_model(best_model_path)
    print(f"✓ Model loaded successfully from: {best_model_path}")
    print(f"  Model size: {os.path.getsize(best_model_path) / (1024*1024):.2f} MB")
    model_loaded = True
    print("  You can skip the training cells and go directly to evaluation/prediction.")
    print("  If you want to retrain, delete the model file first or train again to overwrite.")
elif os.path.exists(final_model_path):
    print("✓ Found saved final model! Loading...")
    model = load_model(final_model_path)
    print(f"✓ Model loaded successfully from: {final_model_path}")
    print(f"  Model size: {os.path.getsize(final_model_path) / (1024*1024):.2f} MB")
    model_loaded = True
    print("  You can skip the training cells and go directly to evaluation/prediction.")
else:
    print("ℹ No saved model found. You need to train the model first.")
    print("  Run the training cells below to create and save a model.")


✓ Found saved best model! Loading...
✓ Model loaded successfully from: models/ai_generated/ai_detector_best.keras
  Model size: 215.20 MB
  You can skip the training cells and go directly to evaluation/prediction.
  If you want to retrain, delete the model file first or train again to overwrite.


## Data Loading Functions


In [ ]:
def load_and_preprocess_image(image_path, img_size=IMG_SIZE):
    """Load and preprocess an image for the model."""
    try:
        img = Image.open(image_path).convert('RGB')
        img = img.resize(img_size)
        img_array = np.array(img) / 255.0  # Normalize to [0, 1]
        return img_array
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        return None


def create_dataset_from_folders(data_dir,max_images_per_class=2000):
    """
    Create dataset from folder structure:
    data_dir/
        real/
        gan/
        diffusion/
    """
    X = []
    y = []
    
    class_folders = {
        'real': 0,
        'gan': 1,
        'diffusion': 2
    }
    
    for class_name, class_label in class_folders.items():
        class_path = os.path.join(data_dir, class_name)
        
        if not os.path.exists(class_path):
            print(f"Warning: {class_path} does not exist. Skipping...")
            continue
            
        image_files = [f for f in os.listdir(class_path) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        print(f"Loading {len(image_files)} images from {class_name}...")
        ctr = 0
        for img_file in image_files:
            img_path = os.path.join(class_path, img_file)
            img_array = load_and_preprocess_image(img_path)
            ctr += 1
            if ctr > 2000:
                break
            if img_array is not None:
                X.append(img_array)
                y.append(class_label)
    
    return np.array(X), np.array(y)


## Load Dataset

**Important:** Make sure your dataset is organized in the folder structure shown below!


In [5]:
# Load dataset from folder structure
# Limit to 10,000 images per class to avoid memory crashes
# Adjust MAX_IMAGES_PER_CLASS based on your RAM:
#   - 5000: ~9GB RAM needed (safer)
#   - 10000: ~18GB RAM needed (recommended if you have 16GB+)
#   - 15000: ~27GB RAM needed (only if you have 32GB+ RAM)

MAX_IMAGES_PER_CLASS = 6000

print(f"Loading dataset with max {MAX_IMAGES_PER_CLASS} images per class...")
print(f"This will load up to {MAX_IMAGES_PER_CLASS * 3} images total (~{MAX_IMAGES_PER_CLASS * 3 * 224 * 224 * 3 * 4 / (1024**3):.1f} GB RAM)\n")

X, y = create_dataset_from_folders(DATASET_DIR, max_images_per_class=MAX_IMAGES_PER_CLASS)

print(f"\n✓ Dataset loaded successfully!")
print(f"Dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Memory used: ~{X.nbytes / (1024**3):.2f} GB")

print(f"\nClass distribution:")
unique, counts = np.unique(y, return_counts=True)
class_names = ['Real', 'GAN', 'Diffusion']
for label, count in zip(unique, counts):
    print(f"  {class_names[label]}: {count} images")


Loading dataset with max 6000 images per class...
This will load up to 18000 images total (~10.1 GB RAM)

Loading 31783 images from real...
Loading 70001 images from gan...
Loading 2851 images from diffusion...

✓ Dataset loaded successfully!
Dataset shape: (6000, 224, 224, 3)
Labels shape: (6000,)
Memory used: ~6.73 GB

Class distribution:
  Real: 2000 images
  GAN: 2000 images
  Diffusion: 2000 images


## Data Preprocessing


In [6]:
# Convert labels to categorical (one-hot encoding)
y_categorical = to_categorical(y, num_classes=NUM_CLASSES)

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_categorical, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Ensure balanced split
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")


Training samples: 4800
Validation samples: 1200


## Model Architecture


In [7]:
def build_ai_detection_model(num_classes=3, img_size=IMG_SIZE):
    """Build EfficientNet-based model for AI-generated image detection."""
    # Load pre-trained EfficientNetB4 (without top layers)
    base_model = EfficientNetB4(
        weights='imagenet',
        include_top=False,
        input_shape=(img_size[0], img_size[1], 3)
    )
    
    # Fine-tune the base model
    base_model.trainable = True
    
    # Add custom classification head
    inputs = tf.keras.Input(shape=(img_size[0], img_size[1], 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='ai_generated_detector')
    return model

# Build and compile model
model = build_ai_detection_model(num_classes=NUM_CLASSES)
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 18s 0us/step


Model: "ai_generated_detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb4 (Functional)     │ (None, 7, 7, 1792)     │    17,673,823 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1792)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       918,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,723,938 (71.43 MB)

 Trainable params: 18,598,731 (70.95 MB)

 Non-trainable params: 125,207 (489.09 KB)

## Callbacks Setup


In [8]:
# Model checkpoint - save best model
checkpoint_path = os.path.join(MODELS_DIR, 'ai_detector_best.keras')
model_checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Early stopping
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate on plateau
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

callbacks = [model_checkpoint, early_stopping, reduce_lr]


## Model Training


In [17]:
# Train the model
history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)


KeyboardInterrupt: 

## Load Model for Inference (Alternative)

If you want to load a saved model separately for inference/evaluation.


In [ ]:
# Load saved model for inference/evaluation
from keras.models import load_model

# Path to saved model (use best or final)
model_path = os.path.join(MODELS_DIR, 'ai_detector_best.keras')

if os.path.exists(model_path):
    print(f"Loading model from {model_path}...")
    loaded_model = load_model(model_path)
    print("✓ Model loaded successfully!")
    print(f"Model summary:")
    loaded_model.summary()
else:
    print(f"❌ Model file not found at: {model_path}")
    print("Please train the model first or check the model path.")


## Model Evaluation


In [ ]:
# Evaluate the model (only run this if you have validation data loaded)
if model_loaded and 'X_val' in locals() and 'y_val' in locals():
    from keras.models import load_model
    
    # Use the loaded model or load best model
    eval_model = model if model_loaded else load_model(os.path.join(MODELS_DIR, 'ai_detector_best.keras'))
    
    # Convert validation data if needed
    if len(y_val.shape) == 1:
        y_val_cat = to_categorical(y_val, num_classes=NUM_CLASSES)
    else:
        y_val_cat = y_val
    
    # Evaluate
    print("Evaluating model on validation set...")
    results = eval_model.evaluate(X_val, y_val_cat, verbose=1)
    print(f"\nValidation Loss: {results[0]:.4f}")
    print(f"Validation Accuracy: {results[1]:.4f}")
    
    # Predictions
    y_pred_proba = eval_model.predict(X_val, verbose=1)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_val_cat, axis=1) if len(y_val_cat.shape) > 1 else y_val
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
elif not model_loaded:
    print("⚠ Please load the dataset and train the model first, or load a saved model.")
else:
    print("⚠ Validation data not loaded. Please run the dataset loading and preprocessing cells first.")


## Image Prediction


In [29]:
def predict_single_image(model, image_path, class_names=CLASS_NAMES, img_size=IMG_SIZE):
    """
    Predict a single image and return class and confidence.
    
    Args:
        model: Trained Keras model
        image_path: Path to image file
        class_names: List of class names
        img_size: Target image size (height, width)
    
    Returns:
        predicted_class: Name of predicted class
        confidence: Confidence score (0-1)
        probabilities: Dictionary with probabilities for each class
    """
    img_array = load_and_preprocess_image(image_path, img_size)
    
    if img_array is None:
        return None, None, None
    
    # Add batch dimension
    img_batch = np.expand_dims(img_array, axis=0)
    
    # Predict
    predictions = model.predict(img_batch, verbose=0)
    predicted_class_idx = np.argmax(predictions[0])
    predicted_class = class_names[predicted_class_idx]
    confidence = predictions[0][predicted_class_idx]
    
    # Get probabilities for all classes
    probabilities = {class_names[i]: float(predictions[0][i]) for i in range(len(class_names))}
    
    return predicted_class, confidence, probabilities


# Example usage (uncomment and provide image path):

list_of_test_images = [f for f in os.listdir('datasets/ai_detection/test')]
# test_image_path = 'path/to/your/test/image.jpg'
# predicted_class, confidence, probs = predict_single_image(model, test_image_path)
# 

for i in list_of_test_images:
    predicted_class, confidence, probs = predict_single_image(model, 'datasets/ai_detection/test/'+i)
    if predicted_class:
        print(f"Predicted class: {predicted_class}")
        print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
        print(f"\nAll class probabilities:")
        for class_name, prob in probs.items():
            print(f"  {class_name}: {prob:.4f} ({prob*100:.2f}%)")
        print("--------------------------------")
        print("Final Verdict:")
        if predicted_class == 'Real':
            print("The image " + i + " is real.")
        else:
            print("The image " + i + " is AI-generated.")
        print("--------------------------------")
    else:
        print("Failed to load or process the image.")


Predicted class: Real
Confidence: 0.9996 (99.96%)

All class probabilities:
  Real: 0.9996 (99.96%)
  GAN: 0.0000 (0.00%)
  Diffusion: 0.0003 (0.03%)
--------------------------------
Final Verdict:
The image Ankita_with_heart_filter.jpg is real.
--------------------------------
Predicted class: GAN
Confidence: 0.9987 (99.87%)

All class probabilities:
  Real: 0.0008 (0.08%)
  GAN: 0.9987 (99.87%)
  Diffusion: 0.0005 (0.05%)
--------------------------------
Final Verdict:
The image fake_9.jpg is AI-generated.
--------------------------------
Predicted class: Diffusion
Confidence: 0.8967 (89.67%)

All class probabilities:
  Real: 0.1033 (10.33%)
  GAN: 0.0000 (0.00%)
  Diffusion: 0.8967 (89.67%)
--------------------------------
Final Verdict:
The image gemini_2.jpg is AI-generated.
--------------------------------
Predicted class: GAN
Confidence: 0.9991 (99.91%)

All class probabilities:
  Real: 0.0005 (0.05%)
  GAN: 0.9991 (99.91%)
  Diffusion: 0.0005 (0.05%)
---------------------------

## Training Curves Visualization (if history is available)


In [ ]:
# Plot training curves if history is available
import pickle

history_path = os.path.join(MODELS_DIR, 'training_history.pkl')

if os.path.exists(history_path):
    with open(history_path, 'rb') as f:
        history_dict = pickle.load(f)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss curve
    axes[0].plot(history_dict['loss'], label='Training Loss', color='blue')
    axes[0].plot(history_dict['val_loss'], label='Validation Loss', color='red')
    axes[0].set_title('Model Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Accuracy curve
    axes[1].plot(history_dict['accuracy'], label='Training Accuracy', color='blue')
    axes[1].plot(history_dict['val_accuracy'], label='Validation Accuracy', color='red')
    axes[1].set_title('Model Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("Training history not found. Train the model first to generate history plots.")


## Confusion Matrix Visualization


In [ ]:
def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion Matrix', cmap=plt.cm.Blues):
    """Plot confusion matrix."""
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        title = 'Normalized ' + title
    
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, fontsize=16)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                horizontalalignment="center",
                color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()


# Generate confusion matrix if validation data is available
if model_loaded and 'X_val' in locals() and 'y_val' in locals():
    from keras.models import load_model
    
    # Use the loaded model or load best model
    eval_model = model if model_loaded else load_model(os.path.join(MODELS_DIR, 'ai_detector_best.keras'))
    
    # Convert validation data if needed
    if len(y_val.shape) == 1:
        y_val_cat = to_categorical(y_val, num_classes=NUM_CLASSES)
    else:
        y_val_cat = y_val
    
    # Predictions
    y_pred_proba = eval_model.predict(X_val, verbose=1)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_val_cat, axis=1) if len(y_val_cat.shape) > 1 else y_val
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    print("Confusion Matrix (Counts):")
    plot_confusion_matrix(cm, CLASS_NAMES)
    
    print("\nConfusion Matrix (Normalized):")
    plot_confusion_matrix(cm, CLASS_NAMES, normalize=True)
elif not model_loaded:
    print("⚠ Please load a saved model or train the model first.")
else:
    print("⚠ Validation data not loaded. Please run the dataset loading cells first.")


## Save Final Model


In [13]:
import pickle

# Save final model
final_model_path = os.path.join(MODELS_DIR, 'ai_detector_final.keras')
model.save(final_model_path)
print(f"✓ Final model saved to: {final_model_path}")

# Save training history
history_path = os.path.join(MODELS_DIR, 'training_history.pkl')
with open(history_path, 'wb') as f:
    pickle.dump(history.history, f)
print(f"✓ Training history saved to: {history_path}")

# Print training summary
print(f"\nTraining Summary:")
print(f"  Epochs completed: {len(history.history['loss'])}")
print(f"  Final training accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"  Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"  Best validation accuracy: {max(history.history['val_accuracy']):.4f}")


✓ Final model saved to: models/ai_generated/ai_detector_final.keras


NameError: name 'history' is not defined